# **Reference Documentation**

In [ ]:
from IPython.display import IFrame

IFrame(
    "https://kubernetes.io/docs/reference/kubectl/",
    width="100%",
    height=700
)

# **Definitions**

#### **Services**
Serivices give clients one stable address for a set of Pods

#### **endpointslices**
Retrieves EndpointSlice resources

#### **TYPE**
ClusterIP means the Service is reachable inside the Kubernetes cluster

#### **CLUSTER-IP:**
The stable internal IP address assigned to the Service
    
#### **kubectl scale**
Changes the desired number of running Pod replicas



# **Commands**

### *Commands*

### *Manifest Files*

#### **Two Deployments Single File Deployment**

*here-document*
cd /home/labex/project/scale-lab
cat <<'EOF' > hostname-web.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: hostname-web
spec:
  replicas: 2
  selector:
    matchLabels:
      app: hostname-web
  template:
    metadata:
      labels:
        app: hostname-web
    spec:
      containers:
        - name: web
          image: busybox:1.36
          imagePullPolicy: IfNotPresent
          command: ["sh", "-c"]
          args:
            - mkdir -p /www; hostname > /www/index.html; exec httpd -f -p 8080 -h /www
          ports:
            - name: http
              containerPort: 8080

---

apiVersion: v1
kind: Service
metadata:
  name: hostname-web
spec:
  selector:
    app: hostname-web
  ports:
    - name: http
      port: 80
      targetPort: http
EOF

---
kubectl apply -f hostname-web.yaml
kubectl rollout status deployment/hostname-web --timeout=60s
kubectl get pods -l app=hostname-web


#### **Nodeport Pod Creation**

**Kubernetes can read several objects from one multi-document YAML file. Each object keeps its own apiVersion, kind, metadata, and spec; a line containing --- separates one YAML document from the next**

*Node port 30080 -> Service port 80 -> Pod port http*

*here-document*

cat <<'EOF' > course-nginx-services.yaml
apiVersion: v1
kind: Service
metadata:
  name: course-nginx
spec:
  type: ClusterIP
  selector:
    app: course-nginx
  ports:
    - name: http
      port: 80
      targetPort: http
      protocol: TCP
--- #three hyphens separate two YAML documents
apiVersion: v1
kind: Service
metadata:
  name: course-nginx-nodeport
spec:
  type: NodePort
  selector:
    app: course-nginx
  ports:
    - name: http
      port: 80
      targetPort: http
      nodePort: 30080 #30080 opens TCP port 30080 on each node
      protocol: TCP
EOF

**Breakdown**

Both Services select Pods labeled app=course-nginx, but they provide different ways to reach them:

a. The ClusterIP Service is for access inside the cluster
b. The NodePort Service additionally exposes the application through port 30080 on each node

#### **Busybox Temp Pod**

kubectl run service-client \
  --image=busybox:1.36 \
  --image-pull-policy=IfNotPresent \
  --restart=Never \
  --rm -i \
  -- wget -qO- http://course-nginx

**Breakdown**

* --image: chooses the container image and 
* --image-pull-policy=IfNotPresent: reuses the cached copy
* --restart=Never: creates a standalone Pod rather than a controller-managed workload
* --rm: removes the Pod after its command exits, and -i keeps command input/output attached
* The -- separator ends kubectl options
    - everything after it is the command inside the container
* wget -qO-: requests the URL quietly and writes the response body to the terminal

#### **ClusterIP Service**

***Service selector -> matching Pod labels -> EndpointSlice addresses -> ready Pods***

**Create the default Service type, ClusterIP**
*It provides a virtual IP and DNS name reachable by workloads inside the cluster*

**here-document**

cat <<'EOF' > course-nginx-service.yaml
apiVersion: v1 #selects the Kubernetes core API
kind: Service # tells Kubernetes to create a Service
metadata: #This names the Service course-nginx
  name: course-nginx
spec:
  type: #ClusterIP #ClusterIP is the default Service type
  selector: #The selector tells the Service which Pods should receive traffic
    app: course-nginx #It selects every Pod with this label
  ports: #This defines how network traffic is forwarded
    - name: http #http gives this port mapping the name http
      port: 80 #the port clients use when connecting to the Service
      targetPort: http # forwards traffic to the container port named http in each selected Pod
      protocol: TCP #specifies that the Service uses TCP networking
EOF

kubectl apply --dry-run=client -f course-nginx-service.yaml
kubectl apply -f course-nginx-service.yaml
kubectl get service course-nginx

**Breakdown**

1. Create an internal Service named course-nginx
2. Assign it a stable internal ClusterIP
3. Find Pods labeled app=course-nginx
4. Accept TCP connections on Service port 80
5. Forward those connections to the selected Pods' named http port

#### **Backend Deployment**

**A Deployment manages a set of identical Pods and helps keep the desired number running**
**The matching app label is the key connection between the Deployment and its Pods**

*here-Document*

cat <<'EOF' > course-nginx-deployment.yaml
apiVersion: apps/v1 #specifies the Kubernetes API version
kind: Deployment #says this object is a Deployment
metadata:
  name: course-nginx #This gives the Deployment the name course-nginx
spec:
  replicas: 2 #This requests two identical Pods running the application
  selector: #This tells the Deployment which Pods it manages
    matchLabels: #This tells the Deployment which Pods it manages
      app: course-nginx #Label name this deployment manages or can be managed by
  template: #This describes how new Pods should be created
    metadata:
      labels: #Each Pod created by this Deployment receives the label app=course-nginx
        app: course-nginx #Label Name
    spec:
      containers: #Each Pod contains one container
        - name: nginx #The container is named nginx
          image: nginx:1.27-alpine #It uses the nginx:1.27-alpine image
          imagePullPolicy: IfNotPresent
          ports:
            - name: http
              containerPort: 80
              protocol: TCP
EOF

kubectl apply -f course-nginx-deployment.yaml
kubectl rollout status deployment/course-nginx --timeout=60s
kubectl get pods -l app=course-nginx -o wide
kubectl get deployment course-nginx

**Overall logic**

The configuration tells Kubernetes:

1. Create a Deployment named course-nginx
2. Keep two Pods running
3. Create Pods labeled app=course-nginx
4. Run an Nginx container in each Pod
5. Use the nginx:1.27-alpine image
6. Identify the container’s HTTP port as TCP port 80

#### **First Pod**

*here-document*

cat <<'EOF' > first-pod.yaml
apiVersion: v1 #Kubernetes' core version 1 API, which is used for Pods
kind: Pod #Declares that this object is a Pod
metadata:
  name: first-nginx #gives the Pod the name
  namespace: default #places it in the default namespace
  labels: # attaches a searchable key-value label to the Pod
    app: first-nginx
spec: #Describes the desired configuration of the Pod
  containers: #Defines a list of containers
    - name: nginx #This Pod contains one container named nginx
      image: nginx:1.27-alpine #Tells Kubernetes to run the NGINX image
      imagePullPolicy: IfNotPresent #Tells Kubernetes to use the image already available on the node
      ports: #Documents the port used by the container:
        - name: http #The port is named http
          containerPort: 80
          protocol: TCP #It uses the TCP network protocol
EOF

#### **Nginx Deployment**

**Deployment -> ReplicaSet -> Pods -> containers**

*here-document*

cat <<'EOF' > course-web-deployment.yaml
apiVersion: apps/v1 #Uses the stable Kubernetes API version for Deployments
kind: Deployment #Tells Kubernetes that this object is a Deployment
metadata: #Provides information about the Deployment
  name: course-web #The deployment name
  labels: #It has the label
    app: course-web #label name
spec: #Requests that Kubernetes keep two matching Pods running
  replicas: 2 #2 is the desired number of Pods
  selector: #This tells the Deployment which Pods it manages
    matchLabels: #The label for the pods it should manage
      app: course-web #label name
  template: #Defines a blueprint for creating each Pod
    metadata:
      labels: #The selector and the Pod template must use matching labels
        app: course-web #Each Pod created from this template receives the label app: course-web, so the Deployment can find and manage it
    spec:
      containers: #Defines one container inside each created Pod
        - name: nginx #Container Name
          image: nginx:1.27-alpine #Image to use
          imagePullPolicy: IfNotPresent #Pull image if its o not present
          ports:
            - name: http #Port name
              containerPort: 80 #NGINX listens on port 80 inside the container
EOF

**Overall logic**

*The manifest asks Kubernetes to:*
1. Create a Deployment named course-web
2. Create two Pods from the provided template
3. Run one NGINX container in each Pod
4. Manage Pods labeled app: course-web
5. Replace a managed Pod if it disappears


### *Command base*


k8s==Container Orchestration
  --Node=Machine with Kubernetes installed where pods are deployed to
  --Cluster=Group of nodes
  --Master=Another node with k8s installed on it that watches over the nodes in the cluster and is responsible for the orchestration of the nodes
  --kubectl=is used to deploy,manage, and maintain the k8s cluste
  --pods=containers are encapsulated into a k8s object called a pod
  --deployment=is a k8s object that manages a group of identical pods

Control Plane
  - kube-apiserver (API): Serves as the cluster’s front door. All administrative commands and resource requests pass through it
	---the entry point for k8s api requests
  - etcd (Key Value Store): Stores all configuration data and the current state of the cluster
  	---If you lose etcd data, you lose the state of the cluster
  - kube-scheduler (SCH): Assigns Pods to Nodes based on resource requirements, constraints, and policies
  - kube-controller-manager (CTLM): Runs a variety of controllers that continually adjust the cluster’s state
  	---Ensuring that the actual state matches the desired state defined by your deployments and configurations

Nodes (Worker Machines)
  - Node: Nodes are where workloads run
  - kubelet (KLT): A node-level agent that communicates with the Control Plane
  	---It ensures Pods are running and reports their status back to the Control Plane
  - kube-proxy: This maintains network fules on the node allow network communication to your pods from inside of outside of the cluster
  - Container Runtime (CR): Software that runs and manages containers (e.g., Docker or containerd)
  	---It creates and manages containerized applications within Pods

Pods
  - Pod: A Pod is the smallest deployable unit in Kubernetes, typically representing a single instance of a running application
   ---Pods can contain one or more containers that share the same network namespace and storage volumes

Services
  - A Service defines Pods policy for how to access them providing IP addresses, DNS names, and load-balancing
  - k8s doesnt talk to a pod directly it talks to a Service instead bc of the temporary nature of pods
	- it give a stable ip (the clusters ip)
	- it gives stable dns configs name
	- it load balances across pods
	- survives pod restarts
	**************************
	Browser / Client
        ↓
	Service (stable)
        ↓
	Pods (ephemeral, changing)
	***************************  
  - Types of Services
	- Cluster IP (default)
		$internal only
		$used for backend communication
	- NodePort
		$exposes service on each node IP
		$lets you access from your machine
	- LoadBalancer
		$cloud-only (or MetalILB in homelabs)

---Hostname and cluster setup
	* hostnamectl set-hostname k8s-controller
	* sudo swapoff -a
	* cat <<EOF | sudo tee /etc/yum.repos.d/kubernetes.repo
	  [kubernetes]
	  name=Kubernetes
	  baseurl=https://pkgs.k8s.io/core:/stable:/v1.30/rpm/
	  enable=1
	  gpgcheck=1
	  repo_gpgcheck=1
	  gpgkey=https://pkgs.k8s.io/core:/stable:/v1.30/rpm/repodata/repomd.xml.key
	  EOF
	* sudo dnf make clean
	* sudo dnf makecache
	* sudo dnf install -y kubectl --disableexludes=kubernetes
	* curl -LO "https://storage.googleapis.com/minikube/releases/latest/minikube-latest.x86_64.rpm"
	* sudo dnf install -y ./minikube-latest.x86_64.rpm
	* minikube version
	* sudo dnf install -y docker-ce docker-ce-cli containerd.io
	* minikube start --driver=docker
	* minikube service nginx-service --url --this command will output the url for accessing the service url

	


When do you use each command?
Use kubectl get pods when you want:
Is my app running?
Is it crashing?
What node is it on?

Example:

kubectl get pods -n test
Use kubectl get svc when you want:
How do I access this app?
What IP/port do I use?
Is it LoadBalancer or NodePort?
kubectl get svc -n test

---
	* curl -LO "https://k8s.io(curl -L -s https://k8s.io)/bin/linux/amd64/kubectl"
	* curl $(kubectl config view --minify -o jsonpath='{.clusters[0].cluster.server}') --insecure
	* kubectl version
	* kubectl version --client
	* kubectl version --output=json
	* kubectl version --output=yaml
		* kubectl version --output=yaml | grep kustomizeVersion <<this is just an example of what we could grep for
	* kubectl explain pod 
	* kubectl create -y --this will show the subcommands for create
	* kubectl create -f namespace.yaml --when pointed to a yaml namespace file it will create the namespace for you 
	* kubectl create deployment nginx-deployment --image=nginx
		* kubectl scale deployment nginx-deployement --relicas=3
			* kubectl get deployments --this will show how much the deployment has scaled up
	* kubectl create job busybox-job --image=busybox -- echo "Hello from k8s" --create job--creates a Job resource instead of a standalone Pod
		* kubectl get jobs
			* kubectl get pods --this should show the pod and its status 
				* kubectl logs job/busybox-job
					* kubectl delete pod nginx-pod
					* kubectl delete deployment nginx-deployment
					* kubectl delete job busybox-job
	* kubectl cluster-info
	* kubectl cluster-info dump
	* kubectl cluster-info dump > cluster-details.txt
	* kubectl run -h --the run command is used to create and run specific images in a pod allowing for customization
	* kubectl run nginx-pod --image=nginx
	* kubectl get componetstatuses
		--scheduler: watches newly created pods with no assigned node and selects a node for them to run on
		--controller manager: runs controller processes which regulate the state of the system i.e. making sure the right number of pod replicas are running
		--etcd: distributed key-value store that acts as k8s backing store for all cluster data
	* kubectl get ns
	* kubectl get ns Namespace_name
	* kubectl get pvc -n zabbix
	* kubectl get nodes
	* kubectl get nodes -o wide
	* kubectl get nodes -n Namespace_name * kubectl describe node minikube
	* kubectl get storageclass
	* kubectl run >> creates a single pod if it dies thats it >> docker run container equivalent 
	* kubectl run hello-minikube
	* kubectl run nginx --image nginx
	* kubectl run nginx --image nginx -n test
	* kubectl run test --image busybox -it --rm -- sh
	* kubectl run test-nginx --image=nginx --port=80
	* kubectl port-forward pod/test-nginx 8080:80 --address 0.0.0.0
	* kubectl expose pod test-nginx --type=NodePort --name=nginxnodeport-svc --port=80
	* kubectl get svc nginx-nodeport-svc
	* kubectl get all -n prod > svc,pods,nodes
	* kubectl get pods -A
	* kubectl get pods -n portainer -w --this will show pods as they are created
	* kubectl exec -h	
	* kubectl exec -it POD_NAME -- /bin/bash
	* kubectl exec nginx-busybox -c busybox -- ls /bin --nginx-busybox is the deployment name -c is the container name
	* kubectl exec -it nginx-busybox -- /bin/bash
	* kubectl exec nginx-env -- sh -c 'echo $MY_VAR'
		* kubectl create deployment nginx --image=nginx --replicas=1
			* kubectl get pods -n defaults
				* kubectl exec -it nginx-433265-wff24 -- /bin/bash
	* kubectl port-forward service/portainer 9443:9443 --namespace portainer --address 0.0.0.0
	* kubectl port-forward pod/portainer-xfiefwwf 9443:9443 -n portainer --address 0.0.0.0
	* kubectl rollout restart deployment/<deployment_name>
	* kubectl rollout restart deployment/<deployment_name> -n <namespace_name>
	* kubectl rollout restart deployment portainer -n portainer --this will restart the pods in the deployment
	* kubectl get pods -o wide >> status,ip,os
	* kubectl get pod uptime-kuma-3wdfvs -n prod -o yaml
	* kubectl get pod uptime-kuma-3wdfv -n prod -o yaml > uptime-kuma3fsfv.yaml
	* kubectl get deployments -A
	* kubectl get deployment uptime-kuma -n prod
	* kubectl get deployment uptime-kuma -n prod -o yaml
	* kubectl get all -A
	* kubectl run nginx --image nginx -n default
	* kubectl run nginx-app --image nginx -n dev 
	* kubectl run nginx-env --image=nginx --env="MY_VAR=my-value"
	* kubectl edit nginx-app >> image: nginx:latest
	* kubectl get pods -o wide -n dev
	* kubectl get pods -A -o wide
	* kubectl get pod webapp -o wide -n dev
	* kubectl get pod webapp -n dev -o yaml
	* kubectl expose pod webapp --type=NodePort --port=80 --this is exposing the vm on port 80 allowing it to be accessed outside of the pod itself
		--traffic then gets forwarded to the correct pods running the application
	* kubectl expose pod test-app --type=LoadBalancer --name=nginx-local-svc --port=80 --target-port=80 -n test
	* kubectl get svc -n test
	* kubectl expose pod test-windows-app --type=LoadBalancer --name=nginx-ext-svc --port=80 --target-port=80 -n default
	* kubectl port-forward pod/web-app 8080:80 -n dev
	* kubectl get svc webapp -n dev
	* kubectl describe -h --this will output all commands and actions tied to the describe command
	* kubectl describe pod webapp -n dev
	* kubectl describe svc nginx-service
	* kubectl create -f webapp.yml
	* kubectl create deployment my-app --image=nginx
	* kubectl create deployment my-app --image=nginx -n test
	* kubectl delete pod webapp -n default
	* kubectl get svc -n prod 
	*****svc explained = “Show me networking entry points for apps in the cluster”
	* kubectl get svc uptime-kuma -n prod
	* kubectl get svc uptime-kuma -n prod -o yaml > uptime-kuma_sevices.yaml
	* kubectl get svc homepage -n prod
	* kubectl apply -f uptime-kuma.yaml >> this is a deployment >> if the pod dies it notices and recreates it >> think docker compose
	* kubectl get svc -A
	* kubectl get ingress -A
	* kubectl get replicasets >> you can create a replicaset its the same but diff. than a deployement...still creates pods but in the rs way
	* kubectl get rs
	* kubectl get rs -n dev
	* kubectl scale rs new-replicaset --replicas=5
	* kubectl scale rs new-replicaset --replicas=2
	* kubectl delete rs replicaset-1 -n prod
	* kubectl api-resources
	* kubectl api-resources | grep replicaset
	* kubectl explain replicaset | grep VERSION
	* kubectl create -f replicaset-definition-1.yaml
	* kubectl create deployment nginx-demo --image=nginx -n dev -o yaml > nginx-demo.yaml
	* kubectl create deployment hello-kubernetes --image=nginx:latest --port=80 
		--'kubectl': command-line tool for interacting with k8s
		--'create deployment': tell k8s to create a new deployment
		--'hello-Kubernetes': is the name we are giving the deployment
		--'--image=nginx:latest': specifies the image we want to use
		--'--port=80': tells k8s that the container will listen on port 80
			* kubectl get deployments 
			* kubectl describe pod hello-kubernetes-883432
			* kubectl port-forward $(kubectl get pods -o name) 8080:80
			* curl http://localhost:8080
			* kubectl delete deployement hello-kubernetes
	* kubeclt delete -f service.yaml -f deployement.yaml
		* kubectl get deployements,services,pods
	* kubectl get deployments -A
	* kubectl get deployments -n test
	* kubeclt get deployment nginx-demo -n test -o yaml > nginx-deployment.yaml
	* kubectl scale deployment nginx-demo --replicas=3
	* kubectl scale deployment nginx-demo -n test --replicas=0
	* kubectl get rs -n test
	* kubectl expose deployment nginx-demo --type=NodePort --port=80 -n test >> this creates a service that points to that deployment exposing it ext.
	* kubectl get svc -n test
	* kubectl get svc -A -o custom-columns='NAMESPACE:.metadata.namespace,NAME:.metadata.name,TYPE:.spec.type,NODEPORTS:.spec.ports[*].nodePort'
	* kubectl get secret --namespace monitoring my-grafana -o jsonpath="{.data.admin-password}" | base64 --decode ; echo
	* kubectl describe secret mysecret -n mynamespace
	* kubectl logs -h
		:syntax: kubectl logs POD_NAME
			* kubectl get pods
			* kubectl logs ngixn-3223r-lfe
	* kubectl logs -f nginx-busybox --this will follow the logs of a pod in real-time
		* kubectl exec -it nginx-busybox -c nginx -- /bin/sh --run this paired with the above command to watch the logs change
			* curl 127.0.0.1 --this will generate traffic and watch the logs flow
	* kubectl logs nginx-fluentd -c nginx --deployment followed by container name 
	* kubectl logs nginx-fluentd -c fluentd --deployment followed by container name
	* kubectl wait --for=condition=Ready pod -l app=nginx

	---
	* minikube config set driver docker
	* minikube start
	* minikube status
	* minikube dashboard --url
	* minikube stop
	* minikube delete
	* minikube pause
	* minikube unpause
	* kubectl apply -f https://raw.githubusercontent.com/kubernetes/dashboard/v2.3.1/aio/deploy/recommended.yaml 
		--kubectl get pods -n kubernetes-dashboard
		--kubectl get ns | grep kubernetes-dashboard
	* vim dashboard-admin.yaml --this needs to be created in order to log in with admin priviledges
---
apiVersion: v1
kind: ServiceAccount
metadata:
  name: admin-user
  namespace: kubernetes-dashboard

apiVersion: rbac.authorization.k8s.io/v1
kind: ClusterRoleBinding
metadata:
  name: admin-user
roleRef:
  apiGroup: rbac.authorization.k8s.io
  kind: ClusterRole
  name: cluster-admin
subjects:
  - kind: ServiceAccount
    name: admin-user
    namespace: kubernetes-dashboard

	* kubectl apply -f dashboard-admin.yaml	
	* kubectl get sa -n kubernetes-dashboard
	* kubectl get clusterrolebinding
	* kubectl -n kubernetes-dashboard create token admin-user



---
 - curl -fsSL https://raw.githubusercontent.com/helm/helm/main/scripts/get-heml-3 | bash
	* helm version
	* helm repo add portainer https://portainer.github.io/k8s/
	* helm repo list
	* helm repo update
	* helm install portainer portainer/portainer --namespace portainer --set service.type=NodePort
		$kubectl get pods -n portainer
		$kubectl get svc -n portainer
	* helm list -A
	* helm status portainer -n portainer
	* helm get values portainer -n portainer
	* helm upgrade portainer portainer/portainer -n portainer
	* helm history portainer -n portainer
	* helm rollback portainer 1 -n portainer
	* helm uninstall portainer -n portainer
	* helm get values portainer -n portainer > portainer-values.yaml
	* helm upgrade portainer portainer/portainer -f portainer-values.yaml

Contstant Commands
	* helm list -A
	* helm status <release>
	* helm history <release>
	* helm upgrade <release>
	* helm rollback <release> <revision>
	* helm uninstall <relase>
	* helm get values <release-name> <namespace> --Name is the value you are looking for in regards to the release-name

📦 What Helm does (simple explanation)
Instead of writing YAML like:
    Deployment
    Service
    ConfigMap
    Ingress
You install “charts” like:
helm install grafana grafana/grafana


🧠 Helm mental model
A Helm chart is:
    “a pre-packaged Kubernetes application”
It contains:
    deployments
    services
    configs
    defaults


cat <<EOF | sudo tee /etc/yum.repos.d/kubernetes.repo
[kubernetes]
name=Kubernetes
baseurl=https://k8s.io
enabled=1
gpgcheck=1
gpgkey=https://k8s.iorepodata/repodata-key.gpg
exclude=kubelet kubeadm kubectl cri-tools kubernetes-cni
EOF

--dnf install -y kubelet kubeadm kubectl --disableexcludes=kubernets


Lab 1:
 --How many nodes are a part of the cluster and what is the version k8s runnong on the nodes
 	* kubectl get nodes

 --What is the k8s control plan url
 	* kubectl cluster-info

 --Create a Pod named nginx-pod using the nginx image
 	* kubectl run nginx-pod --image nginx

Lab 2:
 --How many pods exist in the default namespace 
 	* kubectl get pods -A

 --Create a new pod with the nginx image
 	* kubectl run nginx --image nginx

 --Which nodes are the pods placed on
 	* kubectl get pods -o wide -n default

 --What is the state of the pod webapp
 	* kubectl get pod webapp -o wide -n default

Lab 3:
 --A pod named node-api is deployed and the container within the pod contains an env. variable named APP_COLOR
 	--Identify the value of the APP_COLOR env. variable configured inside the container
	* kubectl describe pod node-api
		--look under Containers >> Environment section for app_color

Lab 4:
 --Create a pod named nginx in the default namespace
	* kubectl run nginx --image nginx
	* kubectl run nginx --image nginx -n default

 --How many pods are running in the default namespace
	* kubeclt get pods -n default
	* kubectl get pods

 --What image are the pods named newpod* running
	* kubectl describe pod newpods-f33EFaz

 --What node are the pods place on
	* kubectl describe pod newpods-fee3fz

 --A new pod named webapp is created...how many containers are part of the webapp pod
	* kubeclt get pods -n default >> Ready 1/2 >> Ready Containers in Pod/Total containers in pod >> 2 containers but only 1 is running

 --What images are used for the webapp pod
	* kubectl describe pod webapp

 --Delete the webapp pod
	* kubectl delete pod webapp

 --Create a pod named redis and the image redis123
	* kubeclt run redis --image redis123 -n default

 --Change the image on the pod to redis
	* kubectl edit redis >> change the value in the image: field >> save and exit >> k8s will auto attempt to apply the update 

Lab 5:

 --How many replicasets exist on the system
	* kubectl get replicaset
	* kubectl get rs

 --Create a replicaset in the default name space and ensure it is running
	* kubectl create -f replicaset-1.yaml >> in hte definition file edit the 'kind' option to create a replicaset instead of a deployement
	* kubectl get rs -n dev -o wide

 --Edit the image the replicaset is pulling and make sure it is running
	* kubectl edit replicaset new-replica-set >> modify the image it is pulling >> save and exit >> delete the pods and the new images will be pulled 

 --Scale the replicaset to 5 pods
	* kubectl edit replicaset new-replicaset >> modify the replicas section to 5 >> save and exit >> the set will be auto updated
	* kubectl scale rs new-replicaset --replicas=5

 --Scale teh replicaset to 2 pods
	* kubectl scale rs new-replicaset --replicas=2

-------------
Lab 6:
--View logs from a specifig pod
cat << EOF | kubectl apply -f -
apiVersion: v1
kind: Pod
metadata:
  name: nginx-busybox
spec:
  containers:
  - name: nginx
    image: nginx
  - name: busybox
    image: busybox
    command:
      - sleep
      - "3600"
EOF
	* kubectl wait --for=condition=Ready pod nginx-busybox
	* kubectl logs nginx-busybox -c busybox
------------
---2Pod Creation && exec
cat << EOF | kubectl apply -f -
apiVersion: v1
kind: Pod
metadata:
  name: nginx-busybox
spec:
  containers:
  - name: nginx
    image: nginx
  - name: busybox
    image: busybox
    command:
      - sleep
      - "3600"
EOF

	* kubectl wait --for=condition=Ready pod nginx-busybox
	* kubectl get pods -n default --the pods name will be there but there are two containers spun up with this deployment
	* kubectl exec nginx-busybox -c busybox -- ls /bin --nginx-busybox is the deployment name -c is the container name
	* kubectl exec -it nginx-busybox -- /bin/bash


### *Cluster Exploration*

#### **k8s Cluster Info**

* minikube version --short
* **kubectl version**
* **kubectl config current-context**
* kubectl config use-context labex-v135
* minikube status -p labex-v135
    --The short option -p means profile is followed by its name
        -host means the Minikube node container is running
* kubectl get nodes
* **kubectl get pods -n kube-system -l tier=control-plane**
    --l tier=control-plane: Filters the results using a label selector
        -It shows only Pods with the label -- tier=control-plane
* **kubectl get pods -n kube-system -l 'k8s-app in (kube-proxy,calico-node)'**
* **kubectl cluster-info**
* **kubectl get nodes -o wide**
* **kubectl describe node labex-v135**
* **kubectl get pods -A**
* **kubectl get deployments -Az**
* **kubectl get services -A**
* **kubectl get all -A**
    -This groups common resources such as Pods, Services, DaemonSets, Deployments, ReplicaSets, and Jobs
        -*ConfigMaps, Secrets, NetworkPolicies, and many others are omitted*

*Scenario:*
*Imagine that you ask Kubernetes to run a web application*
* First, kubectl sends the request to the kube-apiserver, which validates it and stores desired state in etcd
* Next, the kube-scheduler chooses a node for each new Pod
* The kube-controller-manager watches the cluster and works to make actual state match desired state
* Finally, the kubelet on the selected node asks the container runtime to start the Pod's containers. Networking components allow Pods and Services to communicate
* If a Deployment requests three Pods but only two exist, a controller creates the missing Pod

***Use get for a fast table, get -o wide for extra columns, and describe for conditions, capacity, runtime details, and events for one object***

***Namespaces provide scope, Pods run containers, Deployments maintain desired Pod replicas, and Services provide stable network access to changing Pods***

#### Single Pod Deployement **kind:Pod**

* **printf '%s\n' apiVersion kind metadata spec > manifest-fields.txt**
* **kubectl apply --dry-run=client -f first-pod.yaml**
* **kubectl apply --dry-run=client -f first-pod.yaml -o yaml**
    The output option -o means output format
    Supplying yaml asks kubectl to render the parsed object as YAML instead of printing only a one-line result
* **kubectl apply -f first-pod.yaml**
    *kubectl apply*: sends the object to the API server
    *The API server:* stores the desired Pod specification
    *scheduler* and kubelet cooperate to run it on the node
* **kubectl wait --for=condition=Ready pod/first-nginx --timeout=60s**
    *The above command stops successfully when the condition becomes true or fails after 60 seconds*
* **kubectl get pod first-nginx -o wide**
* ***kubectl get pod first-nginx --show-labels***
* **kubectl get pod first-nginx -o jsonpath='Owner: {.metadata.ownerReferences[0].kind}{"\n"}'**

#### ReplicaSets Pod Deployment **kind:Deployment**

* **kubectl apply --dry-run=client -f course-web-deployment.yaml**
* **kubectl diff -f course-web-deployment.yaml || true**
    *Use kubectl diff to compare the manifest with live state*
    *Lines beginning with + represent content that would be added*
    *Lines beginning with - represent content that would be removed*
* **kubectl apply -f course-web-deployment.yaml**
* **kubectl rollout status deployment/course-web --timeout=60s**
    *Asks Kubernetes to show the current progress of a rollout*
    *--timeout=60s:* Sets the maximum waiting time to 60 seconds
    *If it does not become ready in time, the command stops waiting and reports a timeout*
* **kubectl get deployment course-web**
* **kubectl get deployments -A**
* **kubectl get deployment,replicaset,pods -l app=course-web**

#### **Troubleshooting Deployments**

**get -> describe -> events -> repair manifest -> rollout status -> logs -> exec**
* **kubectl apply -f healthy-web.yaml -f broken-web.yaml**
* **kubectl rollout status deployment/healthy-web --timeout=60s**
* **kubectl rollout status deployment/broken-web --timeout=15s || true**
* **kubectl describe pod broken-web-5fcd668857-657vc | grep Failed**
* **kubectl describe pod broken-web-5fcd668857-657vc | grep -i failed**
    - For case insensativity
* **kubectl describe pod broken-web-5fcd668857-657vc | grep -E 'Failed|Error|CrashLoopBackOff|ImagePullBackOff'**
* **kubectl get deployments**
* **kubectl get pods | grep Failed**
* **kubectl get pods -A | grep Failed**
* **kubectl get pods -A | grep -i failed**
* **kubectl get pods -A | grep -E 'Failed|Error|CrashLoopBackOff|ImagePullBackOff'**
* **kubectl get deployments,replicasets,pods -o wide**
    - A **Deployment** reports the desired and available replica counts
    - A **ReplicaSet** carries that desired replica count closer to the Pods
    - A **Pod** reports container readiness and a short status reason
* **kubectl get pods -l app=broken-web -o wide**
* **kubectl get pods -l app=broken-web -o custom-columns='NAME:.metadata.name,READY:.status.containerStatuses[0].ready,WAITING_REASON:.status.containerStatuses[0].state.waiting.reason,NODE:.spec.nodeName'**
* BROKEN_POD=$(kubectl get pods -l app=broken-web -o jsonpath='{.items[0].metadata.name}') -- Creating env var for the pod with the broken...label
* kubectl describe pod "$BROKEN_POD"
    - **Containers → Image shows the exact requested image**
    - **State → Waiting → Reason describes the current container state**
    - **Events records the kubelet's attempts and error messages**
* **kubectl get pod "$BROKEN_POD" -o jsonpath='Image: {.spec.containers[0].image}{"\n"}'**
* **kubectl get events --sort-by='.metadata.creationTimestamp'**
* **kubectl get events -n <namespace> --field-selector type=Warning --sort-by='.metadata.creationTimestamp'**
* **kubectl get events -A --sort-by='.metadata.creationTimestamp'**
* **kubectl get events -A --sort-by='.metadata.creationTimestamp' | grep -Ei "failed|error|warning"**
* **kubectl get events -A --field-selector type=Warning --sort-by='.metadata.creationTimestamp'**
* **grep -n 'image:' healthy-web.yaml broken-web.yaml**
* **sed -i 's/nginx:1.27-alpine-missing/nginx:1.27-alpine/' broken-web.yaml**
* **kubectl apply --dry-run=client -f broken-web.yaml**
* **kubectl diff -f broken-web.yaml || true**
* **kubectl apply -f broken-web.yaml**
* **kubectl rollout status deployment/broken-web --timeout=60s**
* **kubectl get deployments**

Logs
* kubectl exec "$WEB_POD" -- wget -qO- http://127.0.0.1 | head
* kubectl logs "$WEB_POD" --tail=10
* **kubectl logs zabbix-postgresql-0 -n zabbix --tail=10**
* **kubectl exec "$WEB_POD" -- hostname** --inside the container inspection
* **kubectl exec "$WEB_POD" -- nginx -t**
* **kubectl exec "$WEB_POD" -- wget -qO- http://127.0.0.1 >/dev/null && echo "NGINX responded inside the Pod"**

#### **Backend Services**

*backend means a Pod able to receive traffic for a Service*
*A **Service** does not select a Deployment by name; it independently finds Pods whose labels match its selector*

* **kubectl apply -f course-nginx-deployment.yaml**
* **kubectl rollout status deployment/course-nginx --timeout=60s**
* **kubectl get pods -l app=course-nginx -o wide**
* **kubectl get pods -l app=course-nginx --show-labels**
Each Pod has app=course-nginx plus a generated pod-template-hash

* **kubectl get pods -l app=course-nginx -o custom-columns='NAME:.metadata.name,LABEL:.metadata.labels.app,IP:.status.podIP,READY:.status.containerStatuses[0].ready'**
Compare names, labels, and IPs in a compact view
*-o custom-columns creates a table from selected object fields*
* **kubectl get deployment course-nginx -o jsonpath='Selector: {.spec.selector.matchLabels.app}{"\n"}Pod label: {.spec.template.metadata.labels.app}{"\n"}'**
* **kubectl apply --dry-run=client -f course-nginx-service.yaml**
* **kubectl apply -f course-nginx-service.yaml**
* **kubectl get svc -n default**
* **kubectl get service course-nginx**
* **kubectl describe service course-nginx**
* **kubectl describe service | grep -i selector**
* **kubectl get endpointslices -l kubernetes.io/service-name=course-nginx**
* **kubectl get endpointslices -l kubernetes.io/service-name=course-nginx -o jsonpath='{range .items[*].endpoints[*]}{.addresses[0]}{" ready="}{.conditions.ready}{"\n"}{end}'**

#### **Reach the Service by Cluster DNS**

**Kubernetes DNS lets a Pod in the same namespace use the Service name course-nginx instead of remembering its virtual IP**

```
kubectl run service-client \
  --image=busybox:1.36 \
  --image-pull-policy=IfNotPresent \
  --restart=Never \
  --rm -i \
  -- wget -qO- http://course-nginx
```
*Traffic traveled through the Service rather than directly to a chosen Pod IP*

**Run a quieter success check. >/dev/null discards the HTML body, and && prints the message only if the request command succeeds**
```
kubectl run service-client-check \
  --image=busybox:1.36 \
  --image-pull-policy=IfNotPresent \
  --restart=Never \
  --rm -i \
  -- wget -qO- http://course-nginx >/dev/null && echo "ClusterIP Service responded"
```
*The success message proves both DNS resolution and HTTP reachability. The client Pod is temporary; the Service and its two backend Pods remain*

#### **Add a NodePort Service**

* **kubectl apply --dry-run=client -f course-nginx-services.yaml** *--tied to the nodeport manifest file above*
* **kubectl apply -f course-nginx-services.yaml**
* **kubectl get service course-nginx-nodeport**
*PORT(S) column shows 80:30080/TCP: port 80 is the Service port and 30080 is the node-facing port*

* **NODE_IP=$(minikube ip -p labex-v135)**
* **echo "$NODE_IP"**
* **curl -s --retry 5 --retry-connrefused --retry-delay 2 "http://${NODE_IP}:30080" | grep 'Welcome to nginx'**
* **kubectl get services course-nginx course-nginx-nodeport**

*Use ClusterIP for stable communication within the cluster; it is the default and the common foundation for other exposure mechanisms*
*NodePort adds a node-level entry point and is useful for learning, development, or integration with external load balancers*


#### Scale and Load Balance Applications

* **kubectl apply -f hostname-web.yaml** --file is in manifest cell
* **kubectl rollout status deployment/hostname-web --timeout=60s**
* **kubectl get pods -l app=hostname-web**
* **kubectl get deployment hostname-web**
* **kubectl get pods -l app=hostname-web -o wide**
* **kubectl get endpointslices -l kubernetes.io/service-name=hostname-web**
* **kubectl run load-client --image=busybox:1.36 --image-pull-policy=IfNotPresent --restart=Never -- sleep 3600**

Scale Up Declaritively
* **sed -i 's/replicas: 2/replicas: 4/' hostname-web.yaml**
* kubectl diff -f hostname-web.yaml
* **grep -n 'replicas:' hostname-web.yaml**
* kubectl apply -f hostname-web.yaml
* kubectl rollout status deployment/hostname-web --timeout=60s
* kubectl get deployment hostname-web
* kubectl get pods -l app=hostname-web -o wide
* kubectl get endpointslices -l kubernetes.io/service-name=hostname-web -o jsonpath='{range .items[*].endpoints[*]}{.addresses[0]}{" ready="}{.conditions.ready}{"\n"}{end}'
* **kubectl get service hostname-web -o wide**
* **sort -u /tmp/hostname-responses.txt | wc -l**
* **kubectl scale deployment/hostname-web --replicas=2**
* kubectl get pods -l app=hostname-web
* **grep -n 'replicas:' /home/labex/project/scale-lab/hostname-web.yaml**
* **kubectl get deployment hostname-web -o jsonpath='Live replicas: {.spec.replicas}{"\n"}'**
* **sed -i 's/replicas: 4/replicas: 2/' /home/labex/project/scale-lab/hostname-web.yaml**
* kubectl apply -f /home/labex/project/scale-lab/hostname-web.yaml
* **kubectl describe deployment hostname-web | sed -n '/Events:/,$p'**
* **kubectl delete pod load-client --ignore-not-found**


##### **env Var Examples**

```
BROKEN_POD=$(kubectl get pods -l app=broken-web -o jsonpath='{.items[0].metadata.name}')
kubectl get events \
  --field-selector involvedObject.kind=Pod,involvedObject.name="$BROKEN_POD" \
  --sort-by='.metadata.creationTimestamp'
  ```

```
WEB_POD=$(kubectl get pods -l app=broken-web \
  --field-selector=status.phase=Running \
  -o jsonpath='{.items[0].metadata.name}')
echo "$WEB_POD"
```

```
WEB_POD=$(kubectl get pods -l app=broken-web \
  --field-selector=status.phase=Running \
  -o jsonpath='{.items[0].metadata.name}')
```

# **Command Breakdowns**

#### **kubectl run load-client --image=busybox:1.36 --image-pull-policy=IfNotPresent --restart=Never -- sleep 3600**
*kubectl run:* creates a standalone Pod because --restart=Never is set
*The -- separator:* ends kubectl options
*sleep 3600:* is the container command that keeps it alive

---
#### **kubectl get endpointslices -l kubernetes.io/service-name=hostname-web -o jsonpath='{range .items[*].endpoints[*]}{.addresses[0]}{" ready="}{.conditions.ready}{"\n"}{end}'**
*-o wide* displays additional information, such as: Pod IP address
Node where the Pod is running, Readiness and status details
*{range .items[*].endpoints[*]}:* Loops through every endpoint in every matching EndpointSlice
*{.addresses[0]}:* Prints the endpoint's first IP address
*{" ready="}:* Prints the literal text 'ready' followed by condition which will be either true or false
*{.conditions.ready}:*
*{"\n"}:* Prints a newline so each endpoint appears on its own line

---
#### **kubectl scale deployment/hostname-web --replicas=2**
* kubectl scale changes the desired number of running Pod replicas.
* deployment/hostname-web identifies the Deployment to change.
* --replicas=2 tells Kubernetes to maintain two Pods for this Deployment


# **Labs**